<a href="https://colab.research.google.com/github/Anonymus-dev21/IA-model/blob/main/Turism_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U google-generativeai
!pip install openai==0.28
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.5/76.5 kB 5.2 MB/s eta 0:00:00


In [ ]:
import google.generativeai as genai
import openai
import requests
import os
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
from google.colab import userdata
GOOGLE_API_KEY=userdata.get('asistente__turista')
genai.configure(api_key=GOOGLE_API_KEY)

openai.api_key = ""

generation_config = {
  "temperature": 0.45,
  "top_p": 0.95,
  "top_k": 64,
  "max_output_tokens": 8192,
  "response_mime_type": "text/plain",
}

In [ ]:
generation_config = {
  "temperature": 0.45,
  "top_p": 0.95,
  "top_k": 64,
  "max_output_tokens": 8192,
  "response_mime_type": "text/plain",
}

model = genai.GenerativeModel(
  model_name="gemini-1.5-flash",
  generation_config=generation_config,
  system_instruction="Eres un Asistente virtual de turismo el cual ayuda a sus clientes a palnificar el mejor viaje de la manera mas eficiente y hermosa posible.\n Eres un asistente virtual de viajes que ayuda a los usuarios a planificar y reservar destinos, alojamiento y transporte de acuerdo a sus necesidades y presupuesto. \n",
)

history__chat = []

def imggenerator(prompt, history, max_length=900):
    context = ""
    for entry in reversed(history):
        if entry['role'] == 'model':
            context = list(entry['parts'])[0] + " " + context

        if len(context) > max_length // 2:
            break

    enriched_prompt = context + prompt

    if len(enriched_prompt) > max_length:
        enriched_prompt = enriched_prompt[:max_length]

    response = openai.Image.create(
        prompt=enriched_prompt,
        n=1,
        size="512x512"
    )
    image_url = response['data'][0]['url']
    return image_url

# Función principal del chatbot
def chatbot():
    chat_session = model.start_chat(history=history__chat)

    while True:
        user_input = input("Tú: ")

        if user_input.lower() in ["salir", "exit", "no"]:
            print("Viajero Virtual: ¡Espero haberte ayudado! ¡Que tengas un excelente viaje! ✈️😊")
            break

        # Verifica si el input del usuario contiene alguna palabra para generar una imagen
        if any(word in user_input.lower() for word in ["quiero una imagen", "haceme una imagen", "dame una imagen", "imagen"]):
            prompt_for_image = input("Detalla la imagen que deseas generar: ")
            image_url = imggenerator(prompt_for_image, history__chat)
            response = requests.get(image_url)
            img = Image.open(BytesIO(response.content))
            plt.imshow(img)
            plt.axis('off')
            plt.show()
            continue
        response = chat_session.send_message(user_input)
        print("Viajero Virtual: ", response.text)

        # Actualiza el historial de la conversación
        history__chat.append({'role': 'user', 'parts': [user_input]})
        history__chat.append({'role': 'model', 'parts': [response.text]})


In [ ]:
chatbot()

Tú: exit
Viajero Virtual: ¡Espero haberte ayudado! ¡Que tengas un excelente viaje! ✈️😊
